In [34]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/train.csv")

df.head()

,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,gcs_total,pain_score,weight_kg,height_cm,bmi,shock_index,news2_score,disposition,ed_los_hours,triage_acuity
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,14,7,52.3,165.4,19.1,0.725,8,discharged,7.35,2
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,15,-1,73.3,164.4,27.1,0.739,1,discharged,0.70,5
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,15,3,77.1,183.7,22.8,0.798,2,discharged,0.63,5
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,15,7,49.6,172.6,16.6,0.812,2,discharged,1.99,3
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,15,4,71.9,173.4,23.9,0.812,2,transferred,3.58,3


In [35]:
x_train = pd.DataFrame()
df = df.dropna()

# Original Features
x_train['age'] = df['age'].astype(int)
x_train['num_prior_ed_visits_12m'] = df['num_prior_ed_visits_12m'].astype(int)
x_train['num_prior_admissions_12m'] = df['num_prior_admissions_12m'].astype(int)
x_train['num_active_medications'] = df['num_active_medications']
x_train['num_comorbidities'] = df['num_comorbidities']
x_train['systolic_bp'] = df['systolic_bp'].astype(float)
x_train['diastolic_bp'] = df['diastolic_bp'].astype(float)
x_train['mean_arterial_pressure'] = df['mean_arterial_pressure'].astype(float)
x_train['pulse_pressure'] = df['pulse_pressure'].astype(float)
x_train['heart_rate'] = df['heart_rate'].astype(float)
x_train['respiratory_rate'] = df['respiratory_rate'].astype(float)
x_train['temperature_c'] = df['temperature_c'].astype(float)
x_train['spo2'] = df['spo2'].astype(float)
x_train['gcs_total'] = df['gcs_total'].astype(int)
x_train['pain_score'] = df['pain_score'].astype(int)
x_train['bmi'] = df['bmi'].astype(float)
x_train['shock_index'] = df['shock_index'].astype(float)
x_train['news2_score'] = df['news2_score'].astype(int)

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

y = df['triage_acuity'] - 1
X_tr, X_val, y_tr, y_val = train_test_split(x_train, y, test_size=0.1, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_tr)
X_tr, X_val = scaler.transform(X_tr), scaler.transform(X_val)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_tr.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(5, activation='linear')
])
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=75, batch_size=256)
        #   callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])

pred = model.predict(X_val).argmax(axis=1)
print(classification_report(y_val, pred));
print(confusion_matrix(y_val, pred))

Epoch 1/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6496 - loss: 0.7891 - val_accuracy: 0.7965 - val_loss: 0.4714
Epoch 2/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 924us/step - accuracy: 0.8091 - loss: 0.4379 - val_accuracy: 0.8109 - val_loss: 0.4288
Epoch 3/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8183 - loss: 0.4152 - val_accuracy: 0.8142 - val_loss: 0.4178
Epoch 4/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8217 - loss: 0.4063 - val_accuracy: 0.8170 - val_loss: 0.4120
Epoch 5/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 776us/step - accuracy: 0.8233 - loss: 0.4005 - val_accuracy: 0.8192 - val_loss: 0.4082
Epoch 6/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 759us/step - accuracy: 0.8250 - loss: 0.3959 - val_accuracy: 0.8204 - val_loss: 0.4050
Epoch 7/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 863us/step - accuracy: 0.8266 - loss: 0.3918 - val_accuracy: 0.8217 - val_loss: 0.4014
Epoch 8/75
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 899us/step - accuracy: 0.8280 - loss: 0.3881 - val_ac

In [37]:
import uuid
import time
model.save(f"models/{uuid.uuid1()}_{time.time}.keras")